In [30]:
import os
import warnings

from dotenv import load_dotenv

# from langchain.chains.combine_documents import create_stuff_documents_chain
# from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains.llm import LLMChain
from langchain.chains.retrieval import create_retrieval_chain

# from langchain.memory import ChatMessageHistory
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import CharacterTextSplitter
from langchain.tools import tool
from langchain.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

# from langchain.chains import ConversationChain, RetrievalQA # deprecated
# from langchain_community.chat_message_histories import ChatMessageHistory # deprecated
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import (
    CommaSeparatedListOutputParser,
    JsonOutputParser,
    StrOutputParser,
)
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    PromptTemplate,
)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.tools import Tool
from langchain_experimental.utilities import PythonREPL
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.prebuilt import create_react_agent

# from langchain_core.pydantic_v1 import BaseModel, Field
from pydantic import BaseModel



> Deprecateds:
- RetrievalQA -> create_retrieval_chain (topic: RetrievalQA)

- StuffDocumentsChain foi removida (topic: RetrievalQA)

- create_stuff_documents_chain foi removida (topic: RetrievalQA)

- ChatMessageHistory -> InMemoryChatMessageHistory (topic: chat messsage history)

- ConversationChain -> RunnableWithMessageHistory (topic: conversation buffer)


In [2]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"


def warn(*args, **kwargs):
    pass


warnings.warn = warn
warnings.filterwarnings("ignore")

os.environ["ANONYMIZED_TELEMETRY"] = "False"

load_dotenv()

True

In [3]:
model_id = "gpt-4o-mini"

parameters = {
    "max_tokens": 1000,
    "temperature": 0.7,
    "api_key": os.environ.get("OPENAI_API_KEY"),
    # "top_p": 1,
    # "frequency_penalty": 0,
    # "presence_penalty": 0
}

model = ChatOpenAI(model=model_id, **parameters)

# CHAT MODEL

The chat model takes a list of messages as input and returns a new message. All messages have both a role and a content property.

In [8]:
msg = model.invoke("Hello, how are you?")
print(msg.content)

Hello! I'm just a computer program, but I'm here and ready to help you. How can I assist you today?


## CHAT MESSAGE

In [9]:
chat_messages = model.invoke(
    [
        SystemMessage(
            content="You are a helpful AI bot that assists a user in choosing the perfect book to read in one short sentence"
        ),
        HumanMessage(content="recommend a book about dogs"),
    ]
)

chat_messages.content

'Try "The Art of Racing in the Rain" by Garth Stein for a heartfelt story told from a dog\'s perspective.'

# PROMPT TEMPLATES

Prompt templates help translate user input and parameters into instructions for a language model. 

In [10]:
input_variables = ["adjective", "topic"]
template = "Tell me one {adjective} joke about {topic}"

prompt = PromptTemplate(input_variables=input_variables, template=template)

prompt.format(adjective="funny", topic="cars")

'Tell me one funny joke about cars'

In [11]:
prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")
input_ = {
    "adjective": "funny",
    "topic": "cats",
}  # create a dictionary to store the corresponding input to placeholders in prompt template

In [12]:
# chamando via from_templates. Desta forma PromptTemplate nao é instanciado imediatamente.
# A instanciacao acontece em from_template visto se trata de um bound method: <bound method PromptTemplate.from_template of <class 'langchain_core.prompts.prompt.PromptTemplate'>> — callable signature

template = "me conta uma {contar_o_que} sobre {assunto}"
input_variable = {"contar_o_que": "piada", "assunto": "carros"}


prompt = PromptTemplate.from_template(template)
prompt.invoke(input_variable)

StringPromptValue(text='me conta uma piada sobre carros')

## CHAT PROMPT TEMPLATES

Turn chat messages into templates

In [13]:
messages = [("system", "You are a helpful assistant"), ("user", "Tell me a joke about {topic}")]
input_variables = ["topic"]

prompt = ChatPromptTemplate(messages=messages, input_variables=input_variables)
prompt.format(topic="cars")

'System: You are a helpful assistant\nHuman: Tell me a joke about cars'

In [14]:
# another way of calling using from_messages

prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful assistant"), ("user", "Tell me a joke about {topic}")]
)
input_variables = {"topic": "cars"}

prompt.invoke(input_variables)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke about cars', additional_kwargs={}, response_metadata={})])

## MESSAGE PLACEHOLDER

You can use the MessagesPlaceholder prompt template to add a list of messages in a specific location. In `ChatPromptTemplate.from_messages`, you saw how to format two messages, with each message as a string. But what if you want the user to supply a list of messages that you would slot into a particular spot? You can use `MessagesPlaceholder` for this task.

In [15]:
prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful assistant"), MessagesPlaceholder("msgs")]
)

input_variables = {"msgs": [HumanMessage(content="What is the day after Tuesday?")]}

prompt.invoke(input_variables)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the day after Tuesday?', additional_kwargs={}, response_metadata={})])

# OUTPUT PARSERS

Output parsers take the output from an LLM and transform that output to a more suitable format. Parsing the output is very useful when you are using LLMs to generate any form of structured data, or to normalize output from chat models and other LLMs.

In [16]:
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [17]:
joke_query = "Tell me a joke."

output_parser = JsonOutputParser(pydantic_object=Joke)

format_instructions = output_parser.get_format_instructions()


In [18]:
format_instructions

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"setup": {"title": "Setup", "type": "string"}, "punchline": {"title": "Punchline", "type": "string"}}}\n```'

In [19]:
# Create a prompt template that includes:
# 1. Instructions for the LLM to answer the user's query
# 2. Format instructions to ensure the LLM returns properly structured data
# 3. The actual user query placeholder
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={
        "format_instructions": format_instructions
    },  # Static variables set once when creating the prompt.
)

# Create a processing chain that:
# 1. Formats the prompt using the template
# 2. Sends the formatted prompt to the Llama LLM
# 3. Parses the LLM's response using the output parser to extract structured data
chain = prompt | model | output_parser

# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to Llama
# 3. Parse the response into the structure defined by your output parser
# 4. Return the structured result
chain.invoke({"query": joke_query})

{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything!'}

## COMMA SEPARATED LIST PARSER

In [20]:
output_parser = CommaSeparatedListOutputParser()
format_instructions = output_parser.get_format_instructions()
format_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [21]:
prompt = PromptTemplate(
    input_variables=["subject"],
    template="Answer the user query. {format_instructions} \nList five {subject}",
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | model | output_parser
chain.invoke({"subject": "cars"})
chain = prompt | model | output_parser

['Toyota Camry',
 'Honda Accord',
 'Ford Mustang',
 'Chevrolet Malibu',
 'Nissan Altima']

# DOCUMENT

A `Document` object in `LangChain` contains information about some data. A Document object has the following two attributes:

- `page_content`: *`str`*: This attribute holds the content of the document\.
- `metadata`: *`dict`*: This attribute contains arbitrary metadata associated with the document. You can use the metadata to track various details, such as the document ID, the file name, and other details.


In [22]:
doc = Document(
    page_content="""Python is an interpreted high-level general-purpose programming language.
 Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
    metadata={
        "my_document_id": 234234,  # Unique identifier for this document
        "my_document_source": "About Python",  # Source or title information
        "my_document_create_time": 1680013019,  # Unix timestamp for document creation (March 28, 2023)
    },
)

In [23]:
pdf_loader = PyPDFLoader("../../data/LangChain.pdf")
pdf_file = pdf_loader.load()
pdf_file[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-10T11:43:58+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-10T11:43:58+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../../data/LangChain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChainisarapidlyemergingframeworkthatoffersaver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. It provides tools for han-\ndling chat models, integrating retrieva

In [24]:
web_loader = WebBaseLoader(web_path="https://python.langchain.com/v0.2/docs/introduction/")
web_data = web_loader.load()
web_data[0]

Document(metadata={'source': 'https://python.langchain.com/v0.2/docs/introduction/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModel

## TEXT SPLITTERS

One of the most simple examples of making documents better suit your application is to split a long document into smaller chunks that can fit into your model's context window. LangChain has built-in document transformers that ease the process of splitting, combining, filtering, and otherwise manipulating documents.

In [25]:
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(web_data)

Created a chunk of size 1611, which is longer than the specified 200
Created a chunk of size 730, which is longer than the specified 200
Created a chunk of size 835, which is longer than the specified 200


# EMBEDDINGS

Embeddings generate a vector representation for a specified piece or "chunk" of text.  Embeddings offer the advantage of allowing you to conceptualize text within a vector space. Consequently, you can perform operations such as semantic search, where you identify pieces of text that are most similar within the vector space.


In [26]:
embed_param = {"model": "text-embedding-3-small"}

In [27]:
embedding_model = OpenAIEmbeddings(model=embed_param["model"], api_key=parameters["api_key"])

texts = [i.page_content for i in chunks]
embedding_result = embedding_model.embed_documents(texts)


In [28]:
embedding_result[0][:5]

[-0.0194549560546875,
 0.031646728515625,
 0.0046844482421875,
 0.006641387939453125,
 -0.0254669189453125]

## VECTOR STORES

One of the most common ways to store and search over unstructured data is to embed the text data and store the resulting embedding vectors, and then at query time to embed the unstructured query and retrieve the embedding vectors that are 'most similar' to the embedded query.

In [29]:
docs_db = Chroma.from_documents(chunks, embedding_model)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [30]:
type(docs_db)

langchain_community.vectorstores.chroma.Chroma

In [31]:
query = "langchain"
docs = docs_db.similarity_search(query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [32]:
docs[0].page_content

"LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIProductionDe

# RETRIEVERS


A retriever is an interface that returns documents using an unstructured query. Retrievers are more general than a vector store. A retriever does not need to be able to store documents, only to return (or retrieve) them.

## VECTOR STORE-BACKED RETRIEVERS

Vector store retrievers are retrievers that use a vector store to retrieve documents.

In [33]:
# Embeddings pega um texto unstructured e transforma em vetor.
# o chroma salva esse vetor
# o retriever retorna documentos. Ele pega a query , embeda esta query e consulta no banco de dados, retornando documentos.

In [34]:
# docs_db - onde estao guardados os vetores dos pdf
# embedding_model - onde esta declarado o modelo de embedding

query = "langchain"

retriever = docs_db.as_retriever()

result = retriever.invoke(query)
result[0].page_content[:100]

'LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: '

## PARENT DOCUMENT RETRIEVER

The `ParentDocumentRetriever` strikes that balance by splitting and storing small chunks of data. During retrieval, this retriever first fetches the small chunks, but then looks up the parent IDs for the data and returns those larger documents.


Qual a diference entre o InMemoryStore e o Chroma ? 
<details>

Funcionalmente são dois papéis bem diferentes no mesmo fluxo, e ambos guardam dado — só que dados diferentes, pra propósitos diferentes:

vector_store (Chroma) — o índice de busca
Guarda os pedaços pequenos (child chunks) já transformados em vetor. A função dele é achar — dado uma query, ele faz busca por similaridade semântica entre vetores pra descobrir qual pedacinho de texto é mais relevante. Ele é bom em achar precisão fina, mas um pedaço pequeno geralmente não tem contexto suficiente pra ser útil sozinho como resposta.

store / docstore (InMemoryStore) — o arquivo de referência
Guarda os documentos grandes (parent chunks), como texto puro, indexados por um ID — funciona como uma tabela chave-valor simples, sem nenhuma busca semântica envolvida. A função dele é devolver contexto — uma vez que você já sabe qual pedaço é relevante, você usa o ID desse pedaço pra buscar o documento pai inteiro aqui.

Como os dois se conectam:
Quando o documento original é processado, ele primeiro é dividido em pedaços grandes (pais), e cada pedaço grande é dividido em pedaços pequenos (filhos). Cada filho carrega uma referência pro ID do seu pai. Os filhos (pequenos, embedados) vão pro vector_store. Os pais (grandes, texto puro) vão pro store.

Na hora da busca: a query é comparada contra os vetores dos filhos no vector_store — isso acha o pedaço pequeno mais relevante. Mas em vez de devolver esse pedaço pequeno pra você, o retriever pega o ID do pai daquele filho e busca o pai inteiro no store, devolvendo o documento maior, com mais contexto ao redor do trecho relevante.

Resumindo: o vector_store é onde a pergunta encontra o lugar certo; o store é de onde vem a resposta com contexto suficiente. Um sem o outro não fecha o padrão do ParentDocumentRetriever.

In [ ]:
vector_store_ = Chroma(collection_name="demo_1", embedding_function=embedding_model)

child_splitter_ = .load((separator="\n", chunk_size=400, chunk_overlap=50)

parent_splitter_ = CharacterTextSplitter(separator="\n", chunk_size=4000, chunk_overlap=100)

docstore_ = InMemoryStore()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [36]:
retriever = ParentDocumentRetriever(
    vectorstore=vector_store_,
    docstore=docstore_,
    child_splitter=child_splitter_,
    parent_splitter=parent_splitter_,
)

In [37]:
retriever.add_documents(web_data)  # add docs to vectorstore and docstore

Created a chunk of size 1611, which is longer than the specified 400
Created a chunk of size 730, which is longer than the specified 400
Created a chunk of size 835, which is longer than the specified 400


In [38]:
# retrieves and counts the number of parent document IDs stored in the MEMORY document store
len(list(docstore_.yield_keys()))

3

In [39]:
# Next, we verify that the underlying vector store still retrieves the small chunks.
sub_docs = vector_store_.similarity_search(query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [40]:
print(sub_docs[0].page_content)

LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIProductionDep

In [41]:
# And then retrieve the relevant large chunk.
retrieved_docs = retriever.invoke("Langchain")

# Dentro desse invoke, na ordem:
# Embeda a query "Langchain".
# Faz busca por similaridade no Chroma (vector_store_) contra os embeddings dos child chunks → acha o(s) pedaço(s) pequeno(s) mais relevante(s).
# Pega o ID do documento-pai que está nos metadados desses child chunks.
# Busca no docstore_ (InMemoryStore) o texto completo do pai correspondente a esse ID.
# Devolve o documento pai (não o pedaço pequeno).

In [42]:
print(retrieved_docs[0].page_content[:500])

LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangC


# RETRIEVAL QA

> RetrievalQA é Question Answering, não chatbot — cada .invoke() é independente e sem memória.
>> Você pode chamar várias vezes com perguntas diferentes, mas nenhuma sabe da anterior.
Perguntas de acompanhamento ("e ele é gratuito?") não vão entender quam é "ele" por exemplo.
>>> Chatbot de verdade precisa de chat_history entre chamadas (ex: ConversationalRetrievalChain) — o que você tem é a base, não isso ainda.

In [43]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="Responda à pergunta usando o contexto abaixo.\n\nContexto:\n{context}\n\nPergunta: {question}\nResposta:",
)

In [44]:
# Caminho da investigacao:
# 1 - probe=RetrievalQA pede 2 construtores: combine_documents_chain da classe_1 langchain.chains.combine_documents.base.BaseCombineDocumentsChain'> e retriever, que eu ja conheco.
# 2 - uso a classe_1 como probe e nenhum argumento é requerido, a classe é abstrata, logo  orquestrada automaticamente pelo framework e portanto nao é instanciada direto no codigo.
# 3 - Checo a classe_1 em Siblings para descobrir quem implementa esse contrato?"
# Vejo o modulo stuff  stuff.StuffDocumentsChain como um candidato, classe_2, que pede 2 construtores llm_chain (class LLMChain) e document_variable_name (str)
# 4 - corro probe = LLMChain , pede 2 construtores prompt (BasePromptTemplate) e llm (Runnable — ChatOpenAI ). Visto que eu conheco essas duas classes, já tenho todo o caminho esclarecido.

# llm_chain_ = LLMChain(prompt=prompt, llm=model)
# question_ = "O que é langchain"

# combine_docs_chain = StuffDocumentsChain(llm_chain=llm_chain_, document_variable_name="context")

# qa = RetrievalQA(combine_documents_chain=combine_docs_chain, retriever=retriever)
# qa.invoke(question_)

In [45]:
# RetrievalQA is deprecated. The modern, standard way uses LCEL (LangChain Expression Language) via create_retrieval_chain

In [ ]:
# modern way.

prompt_ = PromptTemplate(  # define o template usado no passo 7 (formatar contexto + pergunta pro llm)
    input_variables=[
        "context",
        "input",
    ],  # "context" e "input" sao nomes fixos exigidos pelo contrato do pipeline, nao sao livres
    template="Responda a pergunta usando o contexto abaixo.\n\nContexto:\n{context} \n\nPergunta:\n{input}\nResposta:",  # texto final enviado ao llm, com os placeholders ja preenchidos
)

create_stuff_documents_chain_ = create_stuff_documents_chain(
    llm=model, prompt=prompt_
)  # monta o executor do passo 7: junta os Documents numa string, formata prompt_, chama o llm. ("stuff" = empilhar tudo, context e query),

create_retrieval_chain_ = create_retrieval_chain(  # orquestra retriever + combine_docs_chain numa unica Runnable. Passo 6 popula o context.
    retriever=retriever,  # ParentDocumentRetriever: quando invocado, executa os passos 2-5 (embed -> compara no Chroma -> resgata ID -> busca pai na memoria)
    combine_docs_chain=create_stuff_documents_chain_,  # quando invocado, executa o passo 7 (llm gera resposta a partir do context)
)  # internamente, RunnablePassthrough.assign(context=...) executa o passo 6 (funde o resultado do retriever na chave "context", mantendo o "input" original)

create_retrieval_chain_.invoke(
    {"input": "What is the answer?"}
)  # dispara o passo 1 (query entra como "input") e desencadeia a execucao dos passos 2 a 7 em sequencia

# ==========================================================================
# Sequencia funcional do retriever, passo a passo:
#
# 1) Query chega: invoke({"input": ...}) é o ponto de entrada; a pergunta
#    do usuario fica disponivel sob a chave "input" no dict de trabalho.
#
# 2) Embed da query: o retriever transforma a string da pergunta em vetor,
#    usando o mesmo embedding_model que gerou os vetores dos child chunks.
#
# 3) Compara no Chroma: esse vetor e comparado por similaridade contra os
#    vetores dos child chunks guardados no vector_store_ (Chroma).
#
# 4) Resgata ID: os child chunks mais similares carregam nos metadados
#    o ID do chunk pai a que pertencem.
#
# 5) Busca pai na memoria: o retriever usa esse ID pra buscar o texto
#    completo do chunk pai correspondente no docstore_ (InMemoryStore).
#
# 6) Retorna contexto: create_retrieval_chain funde o resultado do
#    retriever na chave "context", mantendo o "input" original no dict.
#
# 7) LLM gera resposta: combine_docs_chain junta o context numa string,
#    formata a prompt_, envia pro model, e o resultado vira a chave "answer".
#
# Nota: passos 2-6 nao aparecem como linhas separadas neste cell -- eles
# rodam dentro do objeto `retriever` (construido em outro cell) e dentro
# do codigo interno do LangChain, disparados de uma vez pelo .invoke().
# ==========================================================================

In [ ]:
# este codigo cria o contexto sem Retriever, carregando dados direto de docs

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda à pergunta usando apenas o contexto fornecido:\n\n{context}"),
        ("human", "{question}"),
    ]
)


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(i.page_content for i in docs)


_docs = [
    Document(page_content="o preco do BYD Song Premium é de R$200k."),
    Document(page_content="O Marcelo tem dois Song Premium."),
]

_chain = (
    {
        "context": lambda inputs: format_docs(inputs["documentos"]),
        "question": lambda inputs: inputs["question"],
    }
    | prompt
    | model
    | StrOutputParser()
)


response = _chain.invoke({"documentos": _docs, "question": "quanto Marcelo gastou ?"})
print(response)

Marcelo gastou R$400k, pois ele tem dois BYD Song Premium que custam R$200k cada.


# MEMORY

Most LLM applications have a conversational interface. An essential component of a conversation is being able to refer to information introduced earlier in the conversation. At a bare minimum, a conversational system should be able to directly access some window of past messages.


## CHAT MESSAGE HISTORY

One of the core utility classes underpinning most (if not all) memory modules is the `ChatMessageHistory` class. This class is a super lightweight wrapper that provides convenience methods for saving `HumanMessages` and `AIMessages`, and then fetching both types of messages.


In [52]:
chat_history = InMemoryChatMessageHistory()  # no requireds

# chat_history.clear


In [53]:
chat_history.add_message(HumanMessage(content="what is the capital of Brazil ?"))
chat_history.add_message(SystemMessage(content="i am a geography teacher "))

# chat_history.add_message(BaseMessage(content="what is the capital of Brazil ?", type="human")) # Funciona para o add_message que adiciona a mensagem ao historico, mas da erro no invoke que nao reconhece BaseMessage.


In [54]:
chat_history.messages

[HumanMessage(content='what is the capital of Brazil ?', additional_kwargs={}, response_metadata={}),
 SystemMessage(content='i am a geography teacher ', additional_kwargs={}, response_metadata={})]

In [55]:
model.invoke(chat_history.messages)

AIMessage(content='The capital of Brazil is Brasília. It was officially inaugurated as the capital in 1960, designed by the architect Oscar Niemeyer and urban planner Lúcio Costa to promote the development of the interior of the country.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 24, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9709ac0e8b', 'id': 'chatcmpl-EDDKSt5gRpUCekwZuvlA0KH4Tsycw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a006aa-1c87-70a2-953d-6bd36e74b422-0', usage_metadata={'input_tokens': 24, 'output_tokens': 44, 'total_tokens': 68, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_t

# CONVERSATION BUFFER

Conversation buffer memory allows for the storage of messages, which you use to extract messages to a variable. Consider using conversation buffer memory in a chain, setting `verbose=True` so that the prompt is visible.


In [ ]:
query = "quantos anos tem Marcelo"  # 1) Entrada do usuario: a pergunta da vez
id = 45  # 1) Entrada do usuario: session_id, identifica de quem e essa conversa

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Você é um assistente prestativo."),
        MessagesPlaceholder(
            variable_name="history"
        ),  # 5) ponto onde o historico recuperado (passo 4) e injetado
        ("human", "{question}"),  # 5) ponto onde a pergunta nova (passo 1) entra
    ]
)

_chain = (
    prompt | model | StrOutputParser()
)  # 6) a chain "de verdade": so roda depois que historico+pergunta ja estao no prompt

_store = {}  # 3-4) "banco de dados" em memoria: um BaseChatMessageHistory guardado por session_id


def get_session_history(session_id: str):
    # 3) RunnableWithMessageHistory chama essa funcao sozinho, passando o session_id vindo do config
    if session_id not in _store:
        _store[session_id] = (
            InMemoryChatMessageHistory()
        )  # primeira vez desse id: cria historico vazio
    return _store[
        session_id
    ]  # 4) devolve as mensagens ja acumuladas dessa sessao (ou vazio, na primeira vez)


_message_history = RunnableWithMessageHistory(
    runnable=_chain,  # 6) a chain que sera executada com historico + pergunta ja combinados
    get_session_history=get_session_history,  # 3) funcao que localiza/cria o historico da sessao
    input_messages_key="question",  # diz qual chave do dict de input (passo 1) e "a pergunta nova"
    history_messages_key="history",  # diz em qual variavel do prompt (passo 5) o historico deve entrar
)

# 2) RunnableWithMessageHistory intercepta essa chamada antes de rodar a chain de verdade
response = _message_history.invoke(
    {"question": query},  # 1) input da vez, mapeado por input_messages_key
    config={
        "configurable": {"session_id": id}
    },  # 1) canal separado (nao e conteudo da conversa) que identifica a sessao
)
response
# 7) response = resposta da LLM, ja considerando historico + pergunta atual
# 8) Depois dessa chamada, RunnableWithMessageHistory salva sozinho o HumanMessage(query)
#    e o AIMessage(response) em _store[id] -- nao precisa chamar add_message na mao
# 9) Uma proxima invoke() com o mesmo session_id vai reencontrar esse historico ja atualizado
#    (e session_id diferente comeca do zero, com historico proprio)

# ==========================================================================
# Sequencia funcional do conversation buffer, passo a passo:
#
# 1) Entrada do usuario: invoke({"question": ...}, config={"configurable":
#    {"session_id": ...}}) é o ponto de entrada -- pergunta e session_id
#    trafegam em canais separados (input vs config).
#
# 2) RunnableWithMessageHistory intercepta a chamada antes de rodar _chain.
#
# 3) Busca do historico: chama get_session_history(session_id), que consulta
#    (ou cria) o InMemoryChatMessageHistory correspondente em _store.
#
# 4) Recuperacao das mensagens: o historico daquela sessao (lista de
#    HumanMessage/AIMessage anteriores) fica disponivel.
#
# 5) Montagem do input completo: historico + pergunta nova sao injetados
#    no prompt via MessagesPlaceholder("history") e "{question}".
#
# 6) Execucao da chain: _chain (prompt | model | StrOutputParser) roda
#    normalmente, recebendo o prompt ja preenchido.
#
# 7) Resposta da LLM: o model gera a resposta considerando todo o historico.
#
# 8) Persistencia automatica: RunnableWithMessageHistory salva o turno
#    (pergunta + resposta) de volta em _store[session_id].
#
# 9) Proxima chamada: um novo invoke() com o mesmo session_id reencontra
#    esse historico ja atualizado no passo 3, e o ciclo se repete.
# ==========================================================================


In [18]:
_store[45].messages

[HumanMessage(content='quantos anos tem Marcelo', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Desculpe, mas não tenho informações sobre pessoas específicas, como Marcelo, a menos que sejam figuras públicas amplamente conhecidas. Se você puder fornecer mais contexto, ficarei feliz em ajudar de outra forma!', additional_kwargs={}, response_metadata={})]

In [ ]:
_message_history.invoke(
    {"question": "marcelo tem 19 anos e qual o signo dele ?"},
    config={"configurable": {"session_id": id}},
)

'Se Marcelo tem 19 anos e considerarmos que estamos em 2023, isso significa que ele provavelmente nasceu em 2004. Dependendo do mês de nascimento, seu signo pode variar. Aqui estão os signos correspondentes aos meses:\n\n- Áries: 21 de março a 19 de abril\n- Touro: 20 de abril a 20 de maio\n- Gêmeos: 21 de maio a 20 de junho\n- Câncer: 21 de junho a 22 de julho\n- Leão: 23 de julho a 22 de agosto\n- Virgem: 23 de agosto a 22 de setembro\n- Libra: 23 de setembro a 22 de outubro\n- Escorpião: 23 de outubro a 21 de novembro\n- Sagitário: 22 de novembro a 21 de dezembro\n- Capricórnio: 22 de dezembro a 19 de janeiro\n- Aquário: 20 de janeiro a 18 de fevereiro\n- Peixes: 19 de fevereiro a 20 de março\n\nSe você souber o mês de nascimento dele, poderei te ajudar a identificar o signo!'

In [ ]:
_message_history.invoke(
    {"question": "qual idade de marcelo ?"}, config={"configurable": {"session_id": id}}
)

'Se Marcelo tem 19 anos, então a idade dele é 19 anos. Se precisar de mais informações ou ajuda, estou à disposição!'

# LCEL


In [23]:
# Define the templates for each step
location_template = """Your job is to come up with a classic dish from the area that the users suggests.
{location}

YOUR RESPONSE:
"""

dish_template = """Given a meal {meal}, give a short and simple recipe on how to make that dish at home.

YOUR RESPONSE:
"""

time_template = """Given the recipe {recipe}, estimate how much time I need to cook it.

YOUR RESPONSE:
"""

# Create the location chain using LCEL (LangChain Expression Language)
# This chain takes a location and returns a classic dish from that region
location_chain_lcel = (
    PromptTemplate.from_template(location_template)  # Format the prompt with location
    | model  # Send to the LLM
    | StrOutputParser()  # Extract the string response
)

# Create the dish chain using LCEL
# This chain takes a meal name and returns a recipe
dish_chain_lcel = (
    PromptTemplate.from_template(dish_template)  # Format the prompt with meal
    | model  # Send to the LLM
    | StrOutputParser()  # Extract the string response
)

# Create the time estimation chain using LCEL
# This chain takes a recipe and returns an estimated cooking time
time_chain_lcel = (
    PromptTemplate.from_template(time_template)  # Format the prompt with recipe
    | model  # Send to the LLM
    | StrOutputParser()  # Extract the string response
)

# Combine all chains into a single workflow using RunnablePassthrough.assign
# RunnablePassthrough.assign adds new keys to the input dictionary without removing existing ones
overall_chain_lcel = (
    # Step 1: Generate a meal based on location and add it to the input dictionary
    RunnablePassthrough.assign(
        meal=lambda x: location_chain_lcel.invoke({"location": x["location"]})
    )
    # Step 2: Generate a recipe based on the meal and add it to the input dictionary
    | RunnablePassthrough.assign(recipe=lambda x: dish_chain_lcel.invoke({"meal": x["meal"]}))
    # Step 3: Estimate cooking time based on the recipe and add it to the input dictionary
    | RunnablePassthrough.assign(time=lambda x: time_chain_lcel.invoke({"recipe": x["recipe"]}))
)
# Run the chain
result = overall_chain_lcel.invoke({"location": "China"})
print(result)

{'location': 'China', 'meal': 'One classic dish from China is Peking Duck. This iconic dish is known for its crispy skin and tender meat, traditionally served with thin pancakes, hoisin sauce, and sliced scallions. Peking Duck has a rich history and is celebrated for its elaborate preparation and presentation, making it a must-try when exploring Chinese cuisine.', 'recipe': '### Peking Duck Recipe\n\n#### Ingredients:\n- 1 whole duck (about 5-6 lbs)\n- 1 tablespoon salt\n- 2 tablespoons sugar\n- 1 tablespoon soy sauce\n- 1 tablespoon rice vinegar\n- 1 tablespoon hoisin sauce\n- Thin pancakes (store-bought or homemade)\n- Sliced scallions\n- Additional hoisin sauce for serving\n\n#### Instructions:\n\n1. **Prepare the Duck:**\n   - Rinse the duck under cold water and pat it dry with paper towels.\n   - Rub the salt and sugar all over the duck, including the cavity. Let it sit for about 30 minutes.\n\n2. **Air-Dry the Duck:**\n   - Hang the duck in a cool, dry place for about 4-6 hours, 

# TOOLS AND AGENTS

Tools extend an LLM beyond text generation — they let the model trigger real functions (run code, call an API, hit a search index) and read the result back into its reasoning.

An **Agent** is the loop that lets the LLM decide *which* tool to call, when, and how to use the result to keep going or produce a final answer. `create_react_agent` compiles this loop into a small graph with two kinds of steps: the model (decides what to do next) and the tools (execute whatever the model asked for). The model is given each tool's name, argument schema, and description through native tool-calling — it returns a structured `tool_calls` field on its message rather than writing something like `Action: tool_name` as plain text, so there's no free-text format to parse on the way back.


## Tools

A `Tool` wraps a Python function with three things the agent needs: a `name` it can reference, a `func` that actually runs, and a `description` that tells the model when to reach for it. Tool names must match `^[a-zA-Z0-9_-]+$` (no spaces) — tool-calling models expose tools as JSON Schema function definitions, and function names follow identifier rules.

The example below wraps a `PythonREPL` as a calculator tool: instead of asking the LLM to compute an answer directly (LLMs are unreliable at exact arithmetic), the LLM writes Python code and the REPL executes it for an exact result.

In [42]:
# PythonREPL provides a sandboxed namespace where Python code strings can be executed
python_repl = PythonREPL()

# name is the identifier the model calls, func does the work,
# description is what the model reads to decide when this tool applies
python_calculator = Tool(
    name="python_calculator",
    func=python_repl.run,
    description="Useful for math calculations or executing Python code. "
            "The code must explicitly print() the value you want returned — "
            "e.g. print(144**0.5), not just 144**0.5."

)

In [43]:
python_calculator.invoke("a = 3; b = 1; print(a+b)")

'4\n'

### Custom tool via the `@tool` decorator

For tools you write yourself, `@tool` is less boilerplate than the `Tool` class: it reads the function's type hints to build the argument schema and its docstring to build the description.

In [44]:
@tool
def search_weather(location: str):
    """Search for the current weather in the specified location."""
    # In a real application, this would call a weather API
    return f"The weather in {location} is currently sunny and 72°F."

## Toolkit

A toolkit is just a list of tools handed to the agent as one unit — the agent chooses among all of them at each step.

In [45]:
tools = [python_calculator, search_weather]

## Agent

`create_react_agent(model, tools, prompt)` returns a compiled graph (a `CompiledStateGraph`), not a `Runnable` chain — so it's invoked with a `messages` list rather than a plain string or dict. Its state accumulates the full conversation as a list of message objects: the user's message, the model's tool-call requests, the tool outputs, and the model's final reply.

The `prompt` argument becomes the system message that frames the agent's behavior. At each step the model sees the full message history and either calls a tool again or emits a plain-text final answer; the graph loops automatically until the model stops requesting tools, and the answer ends up as the last entry in `result["messages"]`.

> Pylance may show `create_react_agent` here as strikethrough/deprecated: as of `langgraph==1.0`, this function carries a `@deprecated` notice pointing to `from langchain.agents import create_agent`. That replacement only exists in `langchain>=1.0` — this project pins `langchain==0.3.30` (the rest of the notebook, e.g. the `RetrievalQA`/memory sections, is written against that 0.3.x API), so `langchain.agents.create_agent` isn't installed here. Until this project upgrades to `langchain>=1.0`, `langgraph.prebuilt.create_react_agent` is the current working option — the warning is real but not actionable yet in this environment.

In [46]:
agent = create_react_agent(
    model=model,
    tools=tools,
    prompt="You are a helpful assistant with access to tools. Use them when needed to answer accurately.",
)

result = agent.invoke({"messages": [HumanMessage(content="What is the square root of 256?")]})
print(result["messages"][-1].content)

The square root of 256 is 16.


Testing the agent with different query types — it should pick a different tool depending on what's asked:

In [47]:
queries = [
    "What's 345 * 789?",
    "Calculate the square root of 144",
    "What's the weather in Miami?",
    "If it's sunny in Chicago, what would be a good outdoor activity?",
]

for query in queries:
    result = agent.invoke({"messages": [HumanMessage(content=query)]})
    answer = result["messages"][-1].content
    print(f"QUERY: {query}")
    print(f"ANSWER: {answer}\n")

QUERY: What's 345 * 789?
ANSWER: The result of \( 345 \times 789 \) is 272,055.



QUERY: Calculate the square root of 144
ANSWER: The square root of 144 is 12.

QUERY: What's the weather in Miami?
ANSWER: The weather in Miami is currently sunny with a temperature of 72°F.

QUERY: If it's sunny in Chicago, what would be a good outdoor activity?
ANSWER: If it's sunny in Chicago, here are some great outdoor activities you can consider:

1. **Visit Millennium Park**: Enjoy the beautiful gardens, art installations like the Cloud Gate (The Bean), and maybe catch a free concert at the Jay Pritzker Pavilion.

2. **Bike along the Lakefront Trail**: Rent a bike and ride along the scenic 18-mile path that runs along Lake Michigan, offering stunning views of the city skyline.

3. **Explore Lincoln Park Zoo**: This free zoo is a great place to see animals while strolling through the beautiful park.

4. **Take a Boat Tour**: Consider a river architecture boat tour to see Chicago's famous skyscrapers from the water.

5. **Picnic in Grant Park**: Pack a picnic and enjoy the green s

## Exercise: Basic Agent with Custom Tools

Build a small agent that uses two custom tools of your own: a calculator and a text formatter.

**Instructions:**

1. Create two simple tools: a calculator and a text formatter.
2. Set up an agent that can use both.
3. Test the agent with straightforward questions.

**Starter code — fill in the TODO parts:**

In [48]:
# TODO: Implement a simple calculator tool
def calculator(expression: str) -> str:
    """A simple calculator that can add, subtract, multiply, or divide two numbers.
    Input should be a mathematical expression like '2 + 2' or '15 / 3'."""
    try:
        # HINT: Python's eval() will compute a simple expression string
        # Your code here
        pass
    except Exception as e:
        return f"Error calculating: {str(e)}"


# TODO: Implement a text formatting tool
def format_text(text: str) -> str:
    """Format text to uppercase, lowercase, or title case.
    Input should be in the form '[format_type]: [text]',
    where format_type is 'uppercase', 'lowercase', or 'titlecase'."""
    try:
        # HINT: split the input on the first ":" to separate format_type from the text
        # Your code here
        pass
    except Exception as e:
        return f"Error formatting text: {str(e)}"


# TODO: Wrap both functions as Tool objects
# HINT: tool names must match ^[a-zA-Z0-9_-]+$ (no spaces)
exercise_tools = [
    # Your code here
]

# TODO: Build the agent
# HINT: create_react_agent(model=..., tools=..., prompt=...) — a plain string prompt is enough
exercise_agent = None  # Your code here

test_questions = [
    "What is 25 + 63?",
    "Can you convert 'hello world' to uppercase?",
    "Calculate 15 * 7",
    "titlecase: langchain is awesome",
]

# TODO: Run the tests
for question in test_questions:
    print(f"\n===== Testing: {question} =====")
    # Your code here


===== Testing: What is 25 + 63? =====

===== Testing: Can you convert 'hello world' to uppercase? =====

===== Testing: Calculate 15 * 7 =====

===== Testing: titlecase: langchain is awesome =====


<details>
    <summary>Click here for a solution</summary>

```python
def calculator(expression: str) -> str:
    """A simple calculator that can add, subtract, multiply, or divide two numbers.
    Input should be a mathematical expression like '2 + 2' or '15 / 3'."""
    try:
        result = eval(expression)  # fine for a learning exercise; never eval() untrusted input in production
        return f"Result: {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"


def format_text(text: str) -> str:
    """Format text to uppercase, lowercase, or title case.
    Input should be in the form '[format_type]: [text]',
    where format_type is 'uppercase', 'lowercase', or 'titlecase'."""
    try:
        if ":" in text:
            format_type, content = text.split(":", 1)
            format_type = format_type.strip().lower()
            content = content.strip()
        else:
            return f"Missing format. Example: titlecase: {text} -> {text.title()}"

        if format_type == "uppercase":
            return content.upper()
        elif format_type == "lowercase":
            return content.lower()
        elif format_type == "titlecase":
            return content.title()
        else:
            return f"Unknown format {format_type}. Use: uppercase, lowercase, or titlecase"
    except Exception as e:
        return f"Error formatting text: {str(e)}"


exercise_tools = [
    Tool(
        name="calculator",
        func=calculator,
        description="Useful for performing simple math calculations",
    ),
    Tool(
        name="format_text",
        func=format_text,
        description="Useful for formatting text to uppercase, lowercase, or titlecase. Input must be in the form 'format_type: text'",
    ),
]

exercise_agent = create_react_agent(
    model=model,
    tools=exercise_tools,
    prompt="You are a helpful assistant who can use tools to help with simple tasks.",
)

test_questions = [
    "What is 25 + 63?",
    "Can you convert 'hello world' to uppercase?",
    "Calculate 15 * 7",
    "titlecase: langchain is awesome",
]

for question in test_questions:
    print(f"\n===== Testing: {question} =====")
    result = exercise_agent.invoke({"messages": [HumanMessage(content=question)]})
    print(f"Final Answer: {result['messages'][-1].content}")
```

</details>